# Autonomous Code Writer

**Multi-Agent GitHub Workflow** | LangGraph + LangChain | AI Agents Course Project

---

## What it does

You give it a GitHub repo URL. It clones the repo, analyzes it, suggests a useful feature, **asks you** if the idea is good, creates a GitHub issue + branch, implements the change, runs tests, **asks you again** if the result looks right, opens a pull request, and reviews its own diff — leaving the final merge decision to you.

## Two human approval gates

| Gate | When | Your options |
|------|------|--------------|
| **1** | After feature proposal | `approve` → create issue & branch, `reject` → stop, `revise` → regenerate proposal |
| **2** | After implementation | `approve` → push branch & open PR, `reject` → stop, `revise` → redo implementation |

## Five agents

1. **Repository Analyst** — clones & inspects the repo (language, framework, tests)
2. **Feature Strategist** — proposes a conservative, useful feature
3. **GitHub Coordinator** — creates issues, branches, PRs, comments
4. **Implementation Agent** — makes focused code changes on the branch
5. **PR Review Agent** — self-reviews the diff and posts a GitHub comment

## Safety rules

- Never pushes to the default branch
- Never merges pull requests
- Never creates GitHub artifacts without your approval
- API keys loaded from env / Colab Secrets only

## Architecture

```
Request → Clone/Analyze → Propose → [HITL 1] → Issue+Branch → Implement → Verify → [HITL 2] → PR → Review → Done
```

## Assignment checklist

| Requirement | Status |
|-------------|--------|
| ≥2 agents | 5 agents |
| LangGraph stateful workflow | StateGraph + checkpointing |
| LangChain model/tools | OpenAI + tool wrappers |
| Multiple tools | Clone, inspect, GitHub API, test/lint |
| Conversational memory | Memory checkpointer |
| Human-in-the-Loop | 2 approval gates with revise loops |
| `execute_workflow(request)` | Public function |
| 5 test cases | 5 distinct calls on 1 demo repo |
| Approval test | Happy path |
| Revision test | Feature + implementation revision |

---

**Full details:** [README.md](README.md) — architecture diagram, state schema, agent breakdown, requirements mapping, safety docs, Colab setup guide.

In [ ]:
!pip install -q langchain langgraph langchain-openai openai pygithub gitpython langsmith


In [ ]:
!pip install -q python-dotenv
from dotenv import load_dotenv

# Load .env file
load_dotenv()

In [ ]:
import os
import sys

# Attempt to load secrets from Google Colab Secrets (preferred in Colab environment)
try:
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY") or "")
    os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
    os.environ.setdefault("LANGSMITH_API_KEY", userdata.get("LANGSMITH_API_KEY") or "")
except ImportError:
    pass  # Not running in Colab; rely on environment variables

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()

# Set LangSmith tracking
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://aws.api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "autonomous-code-writer")
# Read API key from environment if not already set (do NOT hardcode secrets)
LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "").strip()
if LANGSMITH_API_KEY:
    os.environ.setdefault("LANGSMITH_API_KEY", LANGSMITH_API_KEY)

# Validate secrets without exposing values
missing = []
if not OPENAI_API_KEY:
    missing.append("OPENAI_API_KEY")
if not GITHUB_TOKEN:
    missing.append("GITHUB_TOKEN")

if missing:
    print("ERROR: Missing required secrets. Please set them via environment variables or Colab Secrets.")
    for m in missing:
        print(f"  - {m} is not set")
    print("Setup cannot continue until the secrets are configured.")
    sys.exit(1)
else:
    print("Secrets loaded successfully. (Values are hidden for security.)")


In [ ]:
from langchain_openai import ChatOpenAI

DEFAULT_MODEL_NAME = os.environ.get('OPENAI_MODEL_NAME', 'gpt-5.4-mini-2026-03-17')

llm = ChatOpenAI(
    model=DEFAULT_MODEL_NAME,
    temperature=0.2,
    openai_api_key=OPENAI_API_KEY,
)

print(f'OpenAI model initialized: {DEFAULT_MODEL_NAME}')

In [ ]:
from github import Github

github_client = Github(GITHUB_TOKEN, timeout=30)

try:
    user = github_client.get_user()
    login = user.login
    print(f'GitHub client authenticated as: {login}')
except Exception as e:
    print(f'ERROR: GitHub authentication failed — {e}')
    sys.exit(1)

# Verify repository access with a lightweight metadata call (optional sanity check)
# Replace with your demo repo if you want to test access immediately:
repo = github_client.get_repo('BozhidarN7/ds-algo-visualizer')
print(f'Repo access OK: {repo.full_name}')

In [ ]:
import re
import os
import tempfile
import shutil
from pathlib import Path
from git import Repo
from typing import Dict, List, Any, Optional

# Regex to extract owner/repo from a GitHub URL
GITHUB_URL_RE = re.compile(
    r'https?://github\.com/([A-Za-z0-9_.-]+)/([A-Za-z0-9_.-]+)'
)

# Directories to skip when walking the repo tree
SKIP_DIRS = {
    '.git',
    'node_modules',
    '__pycache__',
    '.venv', 'venv', 'env', 'virtualenv',
    'build', 'dist', 'target', 'out',
    '.pytest_cache', '.mypy_cache', '.tox',
    '.idea', '.vscode',
    'coverage', 'htmlcov',
}

# File extensions to language mapping
EXT_TO_LANGUAGE = {
    '.py': 'Python',
    '.js': 'JavaScript', '.ts': 'TypeScript',
    '.jsx': 'JavaScript (React)', '.tsx': 'TypeScript (React)',
    '.java': 'Java', '.kt': 'Kotlin',
    '.go': 'Go', '.rs': 'Rust',
    '.rb': 'Ruby', '.php': 'PHP',
    '.cpp': 'C++', '.c': 'C', '.h': 'C/C++',
    '.cs': 'C#', '.swift': 'Swift',
    '.r': 'R', '.scala': 'Scala',
    '.html': 'HTML', '.css': 'CSS', '.scss': 'SCSS',
    '.json': 'JSON', '.yaml': 'YAML', '.yml': 'YAML',
    '.md': 'Markdown',
    '.sh': 'Shell', '.sql': 'SQL',
}

# Package manager files to ecosystem mapping
PACKAGE_MANAGER_FILES = {
    'requirements.txt': 'pip', 'setup.py': 'setuptools', 'pyproject.toml': 'modern-python',
    'Pipfile': 'pipenv', 'poetry.lock': 'poetry',
    'package.json': 'npm/yarn/pnpm', 'package-lock.json': 'npm', 'yarn.lock': 'yarn', 'pnpm-lock.yaml': 'pnpm',
    'Cargo.toml': 'cargo', 'Cargo.lock': 'cargo',
    'go.mod': 'go modules', 'go.sum': 'go modules',
    'Gemfile': 'bundler', 'Gemfile.lock': 'bundler',
    'pom.xml': 'maven', 'build.gradle': 'gradle',
    'composer.json': 'composer',
    'pubspec.yaml': 'pub',
    'Podfile': 'cocoapods',
    'Makefile': 'make',
    'CMakeLists.txt': 'cmake',
    'Dockerfile': 'docker', 'docker-compose.yml': 'docker-compose',
}

# Test command detectors by ecosystem
TEST_COMMAND_HINTS = {
    'pip': ['pytest', 'python -m pytest'],
    'setuptools': ['pytest', 'python setup.py test'],
    'modern-python': ['pytest', 'python -m pytest'],
    'pipenv': ['pipenv run pytest'],
    'poetry': ['poetry run pytest'],
    'npm/yarn/pnpm': ['npm test', 'yarn test', 'pnpm test'],
    'npm': ['npm test'],
    'yarn': ['yarn test'],
    'pnpm': ['pnpm test'],
    'cargo': ['cargo test'],
    'go modules': ['go test ./...'],
    'bundler': ['bundle exec rspec', 'bundle exec minitest'],
    'maven': ['mvn test'],
    'gradle': ['gradle test'],
    'composer': ['vendor/bin/phpunit'],
    'pub': ['flutter test', 'dart test'],
    'make': ['make test'],
    'docker': ['docker build .'],
    'docker-compose': ['docker-compose build'],
}

# Lint command detectors by ecosystem
LINT_COMMAND_HINTS = {
    'pip': ['flake8', 'pylint', 'black --check .', 'ruff check .'],
    'setuptools': ['flake8', 'pylint'],
    'modern-python': ['ruff check .', 'black --check .', 'flake8'],
    'pipenv': ['pipenv run flake8'],
    'poetry': ['poetry run ruff check .'],
    'npm/yarn/pnpm': ['npm run lint', 'yarn lint', 'pnpm lint'],
    'npm': ['npm run lint'],
    'yarn': ['yarn lint'],
    'pnpm': ['pnpm lint'],
    'cargo': ['cargo clippy', 'cargo fmt --check'],
    'go modules': ['gofmt -l .', 'golangci-lint run'],
    'bundler': ['rubocop'],
    'maven': ['mvn verify'],
    'gradle': ['gradle check'],
    'composer': ['vendor/bin/phpcs'],
    'pub': ['flutter analyze', 'dart analyze'],
    'make': ['make lint'],
}


In [ ]:
def parse_repository_request(user_request: str) -> Optional[Dict[str, str]]:
    """Extract owner, repo name, and full URL from a user request."""
    match = GITHUB_URL_RE.search(user_request)
    if not match:
        return None
    owner, repo_name = match.group(1), match.group(2)
    repo_url = f'https://github.com/{owner}/{repo_name}'
    return {
        'owner': owner,
        'repo_name': repo_name,
        'repo_url': repo_url,
    }


def clone_repository(repo_url: str, owner: str, repo_name: str) -> str:
    """Clone a repository into a temporary directory. Returns the local path."""
    base_tmp = tempfile.gettempdir()
    clone_dir = os.path.join(base_tmp, f'acw_{owner}_{repo_name}')
    if os.path.exists(clone_dir):
        shutil.rmtree(clone_dir)
    Repo.clone_from(repo_url, clone_dir, depth=1)
    return clone_dir


def list_repository_files(clone_dir: str, max_files: int = 200) -> List[str]:
    """Walk the cloned repo and return relative file paths, skipping irrelevant dirs."""
    files = []
    root = Path(clone_dir)
    for path in root.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(root)
        parts = set(rel.parts[:-1])
        if parts & SKIP_DIRS:
            continue
        if any(part in SKIP_DIRS for part in rel.parts):
            continue
        files.append(str(rel))
        if len(files) >= max_files:
            break
    return sorted(files)


def detect_project_metadata(files: List[str]) -> Dict[str, Any]:
    """Detect language, package manager, and likely test/lint commands from file list."""
    metadata = {
        'languages': set(),
        'package_managers': set(),
        'test_commands': set(),
        'lint_commands': set(),
        'key_files': [],
    }

    for f in files:
        name = os.path.basename(f)
        if name in PACKAGE_MANAGER_FILES:
            metadata['package_managers'].add(PACKAGE_MANAGER_FILES[name])
            metadata['key_files'].append(f)
        _, ext = os.path.splitext(f)
        if ext in EXT_TO_LANGUAGE:
            metadata['languages'].add(EXT_TO_LANGUAGE[ext])
        if name in ('README.md', 'CONTRIBUTING.md', 'LICENSE', '.gitignore', 'Makefile', 'Dockerfile', 'docker-compose.yml', 'tox.ini', 'setup.cfg', 'pytest.ini'):
            if f not in metadata['key_files']:
                metadata['key_files'].append(f)

    for pm in metadata['package_managers']:
        if pm in TEST_COMMAND_HINTS:
            metadata['test_commands'].update(TEST_COMMAND_HINTS[pm])
        if pm in LINT_COMMAND_HINTS:
            metadata['lint_commands'].update(LINT_COMMAND_HINTS[pm])

    metadata['languages'] = sorted(metadata['languages'])
    metadata['package_managers'] = sorted(metadata['package_managers'])
    metadata['test_commands'] = sorted(metadata['test_commands'])
    metadata['lint_commands'] = sorted(metadata['lint_commands'])
    metadata['key_files'] = metadata['key_files'][:20]
    return metadata


def read_key_file_snippets(clone_dir: str, key_files: List[str], max_lines: int = 50) -> Dict[str, str]:
    """Read the first max_lines of each key file for agent context."""
    snippets = {}
    for f in key_files:
        path = Path(clone_dir) / f
        if not path.exists():
            continue
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
                lines = []
                for i, line in enumerate(fh):
                    if i >= max_lines:
                        break
                    lines.append(line.rstrip('\n'))
                snippets[f] = '\n'.join(lines)
        except Exception:
            continue
    return snippets


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

REPOSITORY_ANALYST_PROMPT = """
You are a Repository Analyst. Your job is to understand a codebase from its file tree, metadata, and key file snippets, then produce a concise technical summary.

Guidelines:
- Identify the primary language(s) and framework(s).
- Note the project structure (e.g., monolith, library, service, CLI tool, web app).
- Mention the package manager and build/test setup if detectable.
- List the most important directories and their purposes.
- Flag anything unusual or noteworthy (e.g., no tests, unconventional layout, multiple languages).
- Keep the summary under 250 words.
- Do not include secrets, credentials, or sensitive data.
"""

def build_repo_analysis_prompt(repo_owner: str, repo_name: str, files: List[str], metadata: Dict[str, Any], snippets: Dict[str, str]) -> str:
    lines = [
        f'Repository: {repo_owner}/{repo_name}',
        f'Total files scanned (sample): {len(files)}',
        '',
        'Languages detected:',
    ]
    for lang in metadata.get('languages', []):
        lines.append(f'  - {lang}')
    lines.append('')
    lines.append('Package managers / build tools:')
    for pm in metadata.get('package_managers', []):
        lines.append(f'  - {pm}')
    lines.append('')
    lines.append('Likely test commands:')
    for cmd in metadata.get('test_commands', []):
        lines.append(f'  - {cmd}')
    lines.append('')
    lines.append('Likely lint commands:')
    for cmd in metadata.get('lint_commands', []):
        lines.append(f'  - {cmd}')
    lines.append('')
    lines.append('Key files:')
    for kf in metadata.get('key_files', []):
        lines.append(f'  - {kf}')
    lines.append('')
    lines.append('File tree sample (first 50 files):')
    for f in files[:50]:
        lines.append(f'  {f}')
    if snippets:
        lines.append('')
        lines.append('Key file snippets:')
        for fname, content in snippets.items():
            lines.append(f'\n--- {fname} ---')
            lines.append(content[:800])
    return '\n'.join(lines)


def run_repository_analyst(repo_owner: str, repo_name: str, files: List[str], metadata: Dict[str, Any], snippets: Dict[str, str]) -> str:
    """Invoke the LLM to synthesize a repository summary."""
    prompt = build_repo_analysis_prompt(repo_owner, repo_name, files, metadata, snippets)
    messages = [
        SystemMessage(content=REPOSITORY_ANALYST_PROMPT),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    return str(response.content)


In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from typing import Annotated
from langgraph.graph.message import add_messages

class WorkflowState(TypedDict, total=False):
    user_request: str
    repo_url: str
    repo_owner: str
    repo_name: str
    repo_clone_dir: str
    repo_files: List[str]
    repo_metadata: Dict[str, Any]
    repo_summary: str
    feature_proposal: Dict[str, Any]
    human_decision_gate1: Dict[str, Any]
    issue_url: str
    issue_number: int
    branch_name: str
    implementation_plan: str
    changed_files: List[str]
    diff_summary: str
    verification_results: Dict[str, Any]
    human_decision_gate2: Dict[str, Any]
    pr_url: str
    pr_number: int
    review_summary: str
    review_comment_url: str
    messages: Annotated[List[BaseMessage], add_messages]
    error: str


In [ ]:
def parse_request_node(state: WorkflowState) -> WorkflowState:
    """Parse the user request and extract repository information."""
    user_request = state.get('user_request', '')
    parsed = parse_repository_request(user_request)
    if not parsed:
        return {
            **state,
            'error': 'No valid GitHub repository URL found in the request.',
        }
    return {
        **state,
        'repo_url': parsed['repo_url'],
        'repo_owner': parsed['owner'],
        'repo_name': parsed['repo_name'],
        'messages': state.get('messages', []) + [
            HumanMessage(content=f'Parsed repository: {parsed["owner"]}/{parsed["repo_name"]}')
        ],
    }


def clone_and_inspect_node(state: WorkflowState) -> WorkflowState:
    """Clone the repository and inspect its file tree and metadata."""
    repo_url = state.get('repo_url')
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    if not repo_url or not owner or not repo_name:
        return {**state, 'error': 'Missing repository information for cloning.'}
    try:
        clone_dir = clone_repository(repo_url, owner, repo_name)
        files = list_repository_files(clone_dir)
        metadata = detect_project_metadata(files)
        snippets = read_key_file_snippets(clone_dir, metadata.get('key_files', []))
        return {
            **state,
            'repo_clone_dir': clone_dir,
            'repo_files': files,
            'repo_metadata': metadata,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f'Cloned {len(files)} files into {clone_dir}')
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Failed to clone or inspect repository: {e}',
        }


def analyze_repository_node(state: WorkflowState) -> WorkflowState:
    """Run the Repository Analyst Agent to produce a summary."""
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    files = state.get('repo_files', [])
    metadata = state.get('repo_metadata', {})
    clone_dir = state.get('repo_clone_dir', '')
    if not files or not metadata:
        return {**state, 'error': 'No repository files or metadata available for analysis.'}
    snippets = read_key_file_snippets(clone_dir, metadata.get('key_files', []))
    try:
        summary = run_repository_analyst(owner, repo_name, files, metadata, snippets)
        return {
            **state,
            'repo_summary': summary,
            'messages': state.get('messages', []) + [
                HumanMessage(content='Repository Analyst produced a summary.')
            ],
        }
    except Exception as e:
        return {
            **state,
            'error': f'Repository analysis failed: {e}',
        }


In [ ]:
# Demo: run the repository analysis slice end-to-end on a real repository
# Replace the URL with your own demo repository when running for real.

DEMO_REPO_URL = 'https://github.com/BozhidarN7/ds-algo-visualizer'

demo_state: WorkflowState = {
    'user_request': f'Analyze this repository and suggest a useful improvement: {DEMO_REPO_URL}',
    'messages': [],
}

# Step 1: parse request
demo_state = parse_request_node(demo_state)
if demo_state.get('error'):
    print('Parse error:', demo_state['error'])
else:
    print(f"Parsed: {demo_state['repo_owner']}/{demo_state['repo_name']}")
    print(f"URL: {demo_state['repo_url']}")

# Step 2: clone and inspect
demo_state = clone_and_inspect_node(demo_state)
if demo_state.get('error'):
    print('Clone/inspect error:', demo_state['error'])
else:
    print(f"Cloned to: {demo_state['repo_clone_dir']}")
    print(f"Files found: {len(demo_state['repo_files'])}")
    meta = demo_state['repo_metadata']
    print(f"Languages: {meta.get('languages')}")
    print(f"Package managers: {meta.get('package_managers')}")
    print(f"Test commands: {meta.get('test_commands')}")
    print(f"Lint commands: {meta.get('lint_commands')}")
    print(f"Key files: {meta.get('key_files')}")

# Step 3: analyze with LLM
demo_state = analyze_repository_node(demo_state)
if demo_state.get('error'):
    print('Analysis error:', demo_state['error'])
else:
    print('\n--- Repository Summary ---\n')
    print(demo_state['repo_summary'])

    print('\n--- Detected Stack ---\n')
    meta = demo_state['repo_metadata']
    print(f"Languages: {', '.join(meta.get('languages', [])) or 'None detected'}")
    print(f"Package managers: {', '.join(meta.get('package_managers', [])) or 'None detected'}")
    print(f"Likely tests: {', '.join(meta.get('test_commands', [])) or 'None detected'}")
    print(f"Likely lint: {', '.join(meta.get('lint_commands', [])) or 'None detected'}")


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

FEATURE_STRATEGIST_PROMPT = """
You are a Feature Strategist. Your job is to propose a conservative, useful feature for a repository based on its technical summary and metadata.

Guidelines:
- Generate 2-3 feature ideas relevant to the repository's stack and purpose.
- Prefer small-to-medium features: documentation improvements, developer-experience enhancements, new tests, examples, or focused code additions.
- Avoid risky changes: no authentication rewrites, no database migrations, no deployment changes, no broad dependency upgrades.
- Select the best option and produce a structured proposal.

Output format (strict JSON):
{
  "ideas": ["idea 1", "idea 2", "idea 3"],
  "selected_index": 0,
  "title": "Concise issue title",
  "body": "Detailed feature description, goal, and motivation.",
  "value": "Why this feature is useful.",
  "implementation_scope": "What files or areas will likely change.",
  "risk_level": "low|medium|high",
  "acceptance_criteria": ["criterion 1", "criterion 2"]
}
"""

def build_feature_strategy_prompt(repo_owner: str, repo_name: str, repo_summary: str, metadata: Dict[str, Any]) -> str:
    lines = [
        f'Repository: {repo_owner}/{repo_name}',
        '',
        'Repository summary:',
        repo_summary,
        '',
        'Detected languages:',
    ]
    for lang in metadata.get('languages', []):
        lines.append(f'  - {lang}')
    lines.append('')
    lines.append('Package managers / build tools:')
    for pm in metadata.get('package_managers', []):
        lines.append(f'  - {pm}')
    lines.append('')
    lines.append('Likely test commands:')
    for cmd in metadata.get('test_commands', []):
        lines.append(f'  - {cmd}')
    lines.append('')
    lines.append('Likely lint commands:')
    for cmd in metadata.get('lint_commands', []):
        lines.append(f'  - {cmd}')
    return '\n'.join(lines)


import json

def run_feature_strategist(repo_owner: str, repo_name: str, repo_summary: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    prompt = build_feature_strategy_prompt(repo_owner, repo_name, repo_summary, metadata)
    messages = [
        SystemMessage(content=FEATURE_STRATEGIST_PROMPT),
        HumanMessage(content=prompt),
    ]
    response = llm.invoke(messages)
    content = str(response.content)

    cleaned = content.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]
    if cleaned.startswith('```'):
        cleaned = cleaned[3:]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        proposal = json.loads(cleaned)
    except json.JSONDecodeError:
        proposal = {
            'ideas': ['Feature suggestion based on repository analysis'],
            'selected_index': 0,
            'title': 'Improve repository based on analysis',
            'body': content,
            'value': 'Improves the repository',
            'implementation_scope': 'TBD',
            'risk_level': 'medium',
            'acceptance_criteria': ['Change is implemented and tested'],
        }
    return proposal


In [ ]:
def generate_feature_proposal_node(state: WorkflowState) -> WorkflowState:
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    summary = state.get('repo_summary', '')
    metadata = state.get('repo_metadata', {})
    if not summary:
        return {**state, 'error': 'No repository summary available for feature strategy.'}
    try:
        proposal = run_feature_strategist(owner, repo_name, summary, metadata)
        return {
            **state,
            'feature_proposal': proposal,
            'messages': state.get('messages', []) + [
                HumanMessage(content=f"Feature Strategist selected: {proposal.get('title', 'Untitled')}")
            ],
        }
    except Exception as e:
        return {**state, 'error': f'Feature strategy failed: {e}'}


def human_review_gate1_node(state: WorkflowState) -> WorkflowState:
    proposal = state.get('feature_proposal', {})
    title = proposal.get('title', 'Untitled')
    body = proposal.get('body', '')
    value = proposal.get('value', '')
    scope = proposal.get('implementation_scope', '')
    risk = proposal.get('risk_level', 'unknown')
    criteria = proposal.get('acceptance_criteria', [])

    print('\n' + '=' * 60)
    print('FEATURE PROPOSAL — HUMAN REVIEW REQUIRED')
    print('=' * 60)
    print(f"Title: {title}")
    print(f"Risk: {risk}")
    print('\nDescription:')
    print(body[:600] + ('...' if len(body) > 600 else ''))
    print('\nValue:')
    print(value[:300] + ('...' if len(value) > 300 else ''))
    print('\nImplementation scope:')
    print(scope[:300] + ('...' if len(scope) > 300 else ''))
    print('\nAcceptance criteria:')
    for c in criteria:
        print(f'  - {c}')
    print('\n' + '-' * 60)
    print('Type your decision:')
    print('  approve  — create GitHub issue and branch')
    print('  reject   — stop the workflow')
    print('  revise <feedback> — regenerate proposal with your feedback')
    print('=' * 60 + '\n')

    try:
        user_input = input('Your decision: ').strip()
    except EOFError:
        user_input = ''

    decision = 'pending'
    feedback = ''
    if user_input.lower() == 'approve':
        decision = 'approved'
    elif user_input.lower() == 'reject':
        decision = 'rejected'
    elif user_input.lower().startswith('revise'):
        decision = 'revise'
        feedback = user_input[6:].strip()
    else:
        decision = 'revise'
        feedback = user_input

    return {
        **state,
        'human_decision_gate1': {
            'decision': decision,
            'feedback': feedback,
            'timestamp': None,
        },
        'messages': state.get('messages', []) + [
            HumanMessage(content=f'Human gate 1 decision: {decision}')
        ],
    }


def revise_feature_proposal_node(state: WorkflowState) -> WorkflowState:
    proposal = state.get('feature_proposal', {})
    feedback = state.get('human_decision_gate1', {}).get('feedback', '')
    owner = state.get('repo_owner')
    repo_name = state.get('repo_name')
    summary = state.get('repo_summary', '')
    metadata = state.get('repo_metadata', {})

    revision_prompt = f"""
The previous feature proposal was:

Title: {proposal.get('title', '')}
Body: {proposal.get('body', '')}

Human feedback for revision:
{feedback}

Please regenerate the feature proposal incorporating this feedback.
"""

    messages = [
        SystemMessage(content=FEATURE_STRATEGIST_PROMPT),
        HumanMessage(content=revision_prompt),
    ]
    response = llm.invoke(messages)
    content = str(response.content)

    cleaned = content.strip()
    if cleaned.startswith('```json'):
        cleaned = cleaned[7:]
    if cleaned.startswith('```'):
        cleaned = cleaned[3:]
    if cleaned.endswith('```'):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        proposal = json.loads(cleaned)
    except json.JSONDecodeError:
        proposal = {
            'ideas': ['Revised feature suggestion'],
            'selected_index': 0,
            'title': 'Revised feature proposal',
            'body': content,
            'value': 'Improves the repository',
            'implementation_scope': 'TBD',
            'risk_level': 'medium',
            'acceptance_criteria': ['Change is implemented and tested'],
        }
    return {
        **state,
        'feature_proposal': proposal,
        'human_decision_gate1': {'decision': 'pending', 'feedback': '', 'timestamp': None},
        'messages': state.get('messages', []) + [
            HumanMessage(content=f"Feature Strategist revised proposal: {proposal.get('title', 'Untitled')}")
        ],
    }


def route_gate1_decision(state: WorkflowState) -> str:
    decision = state.get('human_decision_gate1', {}).get('decision', 'pending')
    if decision == 'approved':
        return 'approve'
    elif decision == 'rejected':
        return 'reject'
    else:
        return 'revise'


In [ ]:
# Demo: Feature Strategist + HITL Gate 1
# This cell demonstrates the feature proposal generation and human approval flow.
# In a real run, the graph would pause at gate 1 for human input.

demo_state2: WorkflowState = {
    'user_request': f'Analyze this repository and suggest a useful improvement: {DEMO_REPO_URL}',
    'messages': [],
}

demo_state2 = parse_request_node(demo_state2)
demo_state2 = clone_and_inspect_node(demo_state2)
demo_state2 = analyze_repository_node(demo_state2)

if demo_state2.get('error'):
    print('Error before feature strategy:', demo_state2['error'])
else:
    demo_state2 = generate_feature_proposal_node(demo_state2)
    proposal = demo_state2.get('feature_proposal', {})
    print('\n--- Generated Feature Proposal ---\n')
    print(f"Title: {proposal.get('title')}")
    print(f"Risk: {proposal.get('risk_level')}")
    print(f"Value: {proposal.get('value')}")
    print(f"Scope: {proposal.get('implementation_scope')}")
    print('\nAcceptance criteria:')
    for c in proposal.get('acceptance_criteria', []):
        print(f'  - {c}')
    print('\n--- Full Body ---\n')
    print(proposal.get('body', '')[:800] + ('...' if len(proposal.get('body', '')) > 800 else ''))

    print('\n--- HITL Gate 1 Simulation ---\n')
    print('In the full workflow, the graph would pause here and ask for:')
    print('  approve  — proceed to create issue and branch')
    print('  reject   — stop the workflow')
    print('  revise <feedback> — regenerate the proposal')


In [ ]:
# Issue 4 complete: Feature Strategist Agent + HITL Gate 1 ready.
print('Issue 4 complete: Feature Strategist generates proposals with title, body, value, scope, risk, and acceptance criteria.')
print('HITL Gate 1 supports approve, reject, and revise with feedback loops.')